In [1]:
!pip install -q kagglehub

import os
import glob
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy import stats
from PIL import Image
import kagglehub

# =====================================================================
# 1. DOWNLOAD DATASET & PREPARE SPLITS
# =====================================================================
print("📥 Downloading dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

csv_search = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
df = pd.read_csv(csv_search[0])

all_image_paths = glob.glob(f"{dataset_path}/**/*.jpg", recursive=True)
path_dict = {os.path.basename(p): p for p in all_image_paths}
df['path'] = df['path'].apply(lambda x: path_dict.get(os.path.basename(str(x))))
df = df.dropna(subset=['path'])

# Keep 20% for unseen final testing; remaining 80% is for 3-Fold CV
cv_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Class'], random_state=42)
cv_df = cv_df.reset_index(drop=True)
print(f"✅ Dataset ready! {len(cv_df)} samples allocated for 3-Fold Cross-Validation.")

# =====================================================================
# 2. DATASET CLASS & TRANSFORMS
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CTKidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.dataframe.iloc[idx]['target'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# =====================================================================
# 3. PCSA-KIDNEYNEXT ARCHITECTURE
# =====================================================================
class PCSAModule(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(PCSAModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.conv_spatial_3 = nn.Conv2d(2, 1, kernel_size=3, padding=1, bias=False)
        self.conv_spatial_5 = nn.Conv2d(2, 1, kernel_size=5, padding=2, bias=False)

    def forward(self, x):
        b, c, _, _ = x.size()
        y_avg = self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)
        y_max = self.fc(self.max_pool(x).view(b, c)).view(b, c, 1, 1)
        channel_weights = torch.sigmoid(y_avg + y_max)
        x_out = x * channel_weights

        max_out, _ = torch.max(x_out, dim=1, keepdim=True)
        avg_out = torch.mean(x_out, dim=1, keepdim=True)
        spatial_in = torch.cat([max_out, avg_out], dim=1)
        spatial_weights = torch.sigmoid(self.conv_spatial_3(spatial_in) + self.conv_spatial_5(spatial_in))
        return x_out * spatial_weights

class PCSA_KidneyNeXt_Block(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(PCSA_KidneyNeXt_Block, self).__init__()
        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=stride, padding=1)
        self.avg_pool = nn.AvgPool2d(kernel_size=3, stride=stride, padding=1)
        self.g_conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, padding=1, groups=in_channels, bias=False)
        self.g_conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=5, stride=stride, padding=2, groups=in_channels, bias=False)

        concat_channels = in_channels * 4
        self.compress_conv = nn.Conv2d(concat_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.gelu = nn.GELU()
        self.pcsa = PCSAModule(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        p_max, p_avg = self.max_pool(x), self.avg_pool(x)
        g1, g2 = self.g_conv1(x), self.g_conv2(x)
        out = torch.cat([p_max, p_avg, g1, g2], dim=1)
        out = self.gelu(self.bn(self.compress_conv(out)))
        out = self.pcsa(out)
        return out + self.shortcut(x)

class PCSA_KidneyNeXt(nn.Module):
    def __init__(self, num_classes=4):
        super(PCSA_KidneyNeXt, self).__init__()
        self.stem_conv1 = nn.Conv2d(3, 96, kernel_size=4, stride=4, padding=0, bias=False)
        self.stem_conv2 = nn.Conv2d(3, 96, kernel_size=4, stride=4, padding=0, bias=False)
        self.stem_bn = nn.BatchNorm2d(96)
        self.stem_gelu = nn.GELU()

        self.stage1 = PCSA_KidneyNeXt_Block(96, 96, 1)
        self.stage2 = PCSA_KidneyNeXt_Block(96, 192, 2)
        self.stage3 = PCSA_KidneyNeXt_Block(192, 384, 2)
        self.stage4 = PCSA_KidneyNeXt_Block(384, 768, 2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, x):
        s1, s2 = self.stem_conv1(x), self.stem_conv2(x)
        out = self.stem_gelu(self.stem_bn((s1 + s2) / 2.0))
        out = self.stage4(self.stage3(self.stage2(self.stage1(out))))
        out = self.classifier(torch.flatten(self.global_pool(out), 1))
        return out

# =====================================================================
# 4. 3-FOLD CROSS-VALIDATION EXECUTION (10 EPOCHS PER FOLD)
# =====================================================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
num_epochs = 10
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

pcsa_results = []

print("\n" + "="*60)
print("🚀 INITIALIZING ARCHITECTURE: PCSA-KidneyNeXt (3-Fold CV)")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(cv_df, cv_df['Class']):
    print(f"\n>>> Starting Fold {fold}/3 for PCSA-KidneyNeXt <<<")
    print("-" * 50)

    fold_train_df = cv_df.iloc[train_idx]
    fold_val_df = cv_df.iloc[val_idx]

    fold_train_dataset = CTKidneyDataset(fold_train_df, transform=train_transforms)
    fold_val_dataset = CTKidneyDataset(fold_val_df, transform=test_val_transforms)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize fresh model for each fold
    model = PCSA_KidneyNeXt(num_classes=4).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss, train_corrects, train_size = 0.0, 0, 0
        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_size += inputs.size(0)

        epoch_train_loss = train_loss / train_size
        epoch_train_acc = train_corrects.double().item() / train_size

        # Validation Phase
        model.eval()
        val_loss, val_corrects, val_size = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_size += inputs.size(0)

        epoch_val_loss = val_loss / val_size
        epoch_val_acc = val_corrects.double().item() / val_size

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

        pcsa_results.append({
            "Model": "PCSA_KidneyNeXt",
            "Fold": fold,
            "Epoch": epoch + 1,
            "Train_Loss": epoch_train_loss,
            "Train_Acc": epoch_train_acc,
            "Val_Loss": epoch_val_loss,
            "Val_Acc": epoch_val_acc
        })

    fold += 1

# =====================================================================
# 5. CALCULATE STATISTICAL METRICS (MEAN, STD DEV, P-VALUE)
# =====================================================================
df_pcsa = pd.DataFrame(pcsa_results)

# Paired t-test across all 30 epoch data points (3 folds x 10 epochs)
t_stat, p_value = stats.ttest_rel(df_pcsa['Train_Loss'], df_pcsa['Val_Loss'])

print("\n" + "📊"*20)
print("PCSA-KIDNEYNEXT 3-FOLD CV STATISTICAL METRICS")
print("📊"*20)
print(f"Model Name          : PCSA-KidneyNeXt")
print(f"Mean Val Accuracy   : {df_pcsa['Val_Acc'].mean()*100:.2f}%")
print(f"Std Dev Val Acc     : {df_pcsa['Val_Acc'].std():.4f}")
print(f"Mean Val Loss       : {df_pcsa['Val_Loss'].mean():.4f}")
print(f"Std Dev Val Loss    : {df_pcsa['Val_Loss'].std():.4f}")
print(f"p-value (Train/Val) : {p_value:.4e}")

# Save results for later comparison with baselines
df_pcsa.to_csv("pcsa_3fold_results.csv", index=False)
print("\n✅ Results saved to 'pcsa_3fold_results.csv'!")

📥 Downloading dataset via kagglehub...


100%|██████████| 1.52G/1.52G [01:32<00:00, 17.6MB/s]

Extracting files...


✅ Dataset ready! 9956 samples allocated for 3-Fold Cross-Validation.

🚀 INITIALIZING ARCHITECTURE: PCSA-KidneyNeXt (3-Fold CV)

>>> Starting Fold 1/3 for PCSA-KidneyNeXt <<<
--------------------------------------------------
Epoch 01/10 | Train Loss: 0.7032 - Train Acc: 0.7166 | Val Loss: 0.9851 - Val Acc: 0.6833
Epoch 02/10 | Train Loss: 0.4161 - Train Acc: 0.8445 | Val Loss: 1.1751 - Val Acc: 0.5884
Epoch 03/10 | Train Loss: 0.2992 - Train Acc: 0.8884 | Val Loss: 1.3568 - Val Acc: 0.5020
Epoch 04/10 | Train Loss: 0.2079 - Train Acc: 0.9233 | Val Loss: 1.4481 - Val Acc: 0.6345
Epoch 05/10 | Train Loss: 0.1577 - Train Acc: 0.9430 | Val Loss: 0.2915 - Val Acc: 0.8861
Epoch 06/10 | Train Loss: 0.1089 - Train Acc: 0.9593 | Val Loss: 0.1611 - Val Acc: 0.9382
Epoch 07/10 | Train Loss: 0.0830 - Train Acc: 0.9684 | Val Loss: 0.2602 - Val Acc: 0.8982
Epoch 08/10 | Train Loss: 0.0623 - Train Acc: 0.9783 | Val Loss: 0.4788 - Val Acc: 0.8252
Epoch 09/10 | Train Loss: 0.0649 - Train Acc: 0.9786 | 

In [2]:
!pip install -q kagglehub

import os
import glob
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy import stats
from PIL import Image
import kagglehub

# =====================================================================
# 1. DOWNLOAD DATASET & PREPARE SPLITS
# =====================================================================
print("📥 Downloading dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

csv_search = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
df = pd.read_csv(csv_search[0])

all_image_paths = glob.glob(f"{dataset_path}/**/*.jpg", recursive=True)
path_dict = {os.path.basename(p): p for p in all_image_paths}
df['path'] = df['path'].apply(lambda x: path_dict.get(os.path.basename(str(x))))
df = df.dropna(subset=['path'])

# Keep 20% for unseen final testing; remaining 80% is for 3-Fold CV
cv_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Class'], random_state=42)
cv_df = cv_df.reset_index(drop=True)
print(f"✅ Dataset ready! {len(cv_df)} samples allocated for 3-Fold Cross-Validation.")

# =====================================================================
# 2. DATASET CLASS & TRANSFORMS
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CTKidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.dataframe.iloc[idx]['target'])
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# =====================================================================
# 3. RESNET50 MODEL LOADER (WITH FROZEN BACKBONE)
# =====================================================================
def build_resnet50(num_classes=4):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    # Freeze pre-trained feature extractor backbone
    for param in model.parameters():
        param.requires_grad = False
    # Replace final fully connected layer (automatically gets requires_grad=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

# =====================================================================
# 4. 3-FOLD CROSS-VALIDATION EXECUTION (10 EPOCHS PER FOLD)
# =====================================================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
num_epochs = 10
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

resnet_results = []

print("\n" + "="*60)
print("🚀 INITIALIZING ARCHITECTURE: ResNet50 (3-Fold CV)")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(cv_df, cv_df['Class']):
    print(f"\n>>> Starting Fold {fold}/3 for ResNet50 <<<")
    print("-" * 50)

    fold_train_df = cv_df.iloc[train_idx]
    fold_val_df = cv_df.iloc[val_idx]

    fold_train_dataset = CTKidneyDataset(fold_train_df, transform=train_transforms)
    fold_val_dataset = CTKidneyDataset(fold_val_df, transform=test_val_transforms)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize fresh model for each fold
    model = build_resnet50(num_classes=4).to(device)
    # Optimize only trainable parameters
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, weight_decay=1e-4)

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss, train_corrects, train_size = 0.0, 0, 0
        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_size += inputs.size(0)

        epoch_train_loss = train_loss / train_size
        epoch_train_acc = train_corrects.double().item() / train_size

        # Validation Phase
        model.eval()
        val_loss, val_corrects, val_size = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_size += inputs.size(0)

        epoch_val_loss = val_loss / val_size
        epoch_val_acc = val_corrects.double().item() / val_size

        print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.4f}")

        resnet_results.append({
            "Model": "ResNet50",
            "Fold": fold,
            "Epoch": epoch + 1,
            "Train_Loss": epoch_train_loss,
            "Train_Acc": epoch_train_acc,
            "Val_Loss": epoch_val_loss,
            "Val_Acc": epoch_val_acc
        })

    fold += 1

# =====================================================================
# 5. CALCULATE STATISTICAL METRICS (MEAN, STD DEV, P-VALUE)
# =====================================================================
df_resnet = pd.DataFrame(resnet_results)

# Paired t-test across all 30 epoch data points (3 folds x 10 epochs)
t_stat, p_value = stats.ttest_rel(df_resnet['Train_Loss'], df_resnet['Val_Loss'])

print("\n" + "📊"*20)
print("RESNET50 3-FOLD CV STATISTICAL METRICS")
print("📊"*20)
print(f"Model Name          : ResNet50")
print(f"Mean Val Accuracy   : {df_resnet['Val_Acc'].mean()*100:.2f}%")
print(f"Std Dev Val Acc     : {df_resnet['Val_Acc'].std():.4f}")
print(f"Mean Val Loss       : {df_resnet['Val_Loss'].mean():.4f}")
print(f"Std Dev Val Loss    : {df_resnet['Val_Loss'].std():.4f}")
print(f"p-value (Train/Val) : {p_value:.4e}")

# Save results to CSV
df_resnet.to_csv("resnet50_3fold_results.csv", index=False)
print("\n✅ Results saved to 'resnet50_3fold_results.csv'!")

📥 Downloading dataset via kagglehub...
Using Colab cache for faster access to the 'ct-kidney-dataset-normal-cyst-tumor-and-stone' dataset.
✅ Dataset ready! 9956 samples allocated for 3-Fold Cross-Validation.

🚀 INITIALIZING ARCHITECTURE: ResNet50 (3-Fold CV)

>>> Starting Fold 1/3 for ResNet50 <<<
--------------------------------------------------
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


Epoch 01/10 | Train Loss: 0.9419 - Train Acc: 0.6648 | Val Loss: 0.7436 - Val Acc: 0.7608
Epoch 02/10 | Train Loss: 0.6700 - Train Acc: 0.7760 | Val Loss: 0.6125 - Val Acc: 0.7915
Epoch 03/10 | Train Loss: 0.5673 - Train Acc: 0.8144 | Val Loss: 0.5484 - Val Acc: 0.7969
Epoch 04/10 | Train Loss: 0.5066 - Train Acc: 0.8323 | Val Loss: 0.5143 - Val Acc: 0.8259
Epoch 05/10 | Train Loss: 0.4559 - Train Acc: 0.8526 | Val Loss: 0.4353 - Val Acc: 0.8596
Epoch 06/10 | Train Loss: 0.4155 - Train Acc: 0.8656 | Val Loss: 0.4317 - Val Acc: 0.8430
Epoch 07/10 | Train Loss: 0.4047 - Train Acc: 0.8665 | Val Loss: 0.4033 - Val Acc: 0.8873
Epoch 08/10 | Train Loss: 0.3837 - Train Acc: 0.8755 | Val Loss: 0.3967 - Val Acc: 0.8602
Epoch 09/10 | Train Loss: 0.3530 - Train Acc: 0.8914 | Val Loss: 0.3669 - Val Acc: 0.8732
Epoch 10/10 | Train Loss: 0.3483 - Train Acc: 0.8897 | Val Loss: 0.3738 - Val Acc: 0.8726

>>> Starting Fold 2/3 for ResNet50 <<<
--------------------------------------------------
Epoch 01/